# Adaptive Moving Average Forecasting Demo

This demo implements and evaluates an adaptive moving average forecasting method with dynamic window sizing on synthetic time series benchmarks, comparing its predictive accuracy against naive persistence and fixed moving average baselines.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import numpy as np
import os
import urllib.request
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

## Data Loading

Load the curated mini demo dataset from GitHub (with local fallback).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-230d6e-robust-temporal-smoothing-evaluating-mov/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print(f"Loaded {len(data['datasets']} datasets successfully.")

## Configuration

Define parameters for fixed and adaptive moving average forecasting.

In [ ]:
# Configuration parameters
FIXED_WINDOW = 3
ADAPTIVE_SHORT_WINDOW = 2
ADAPTIVE_LONG_WINDOW = 6
ADAPTIVE_DEFAULT_WINDOW = 3
VOLATILITY_HIGH_THRESHOLD = 1.0
VOLATILITY_LOW_THRESHOLD = 0.2

## Evaluation Pipeline

Run naive, fixed moving average, and adaptive moving average forecasting models across all datasets.

In [ ]:
datasets_output = []

for ds in data['datasets']:
    ds_name = ds['dataset']
    examples_out = []
    
    for ex in ds['examples']:
        inp = json.loads(ex['input'])
        target = float(ex['output'])
        
        # Naive forecast: last value
        naive_pred = inp[-1]
        
        # Fixed MA
        if len(inp) >= FIXED_WINDOW:
            fixed_pred = np.mean(inp[-FIXED_WINDOW:])
        else:
            fixed_pred = np.mean(inp)
        
        # Adaptive MA: adjust window based on recent volatility
        recent = inp[-5:] if len(inp) >= 5 else inp
        vol = np.var(recent) if len(recent) > 1 else 0.0
        
        if vol > VOLATILITY_HIGH_THRESHOLD:
            adap_window = ADAPTIVE_SHORT_WINDOW
        elif vol < VOLATILITY_LOW_THRESHOLD:
            adap_window = ADAPTIVE_LONG_WINDOW
        else:
            adap_window = ADAPTIVE_DEFAULT_WINDOW
            
        if len(inp) >= adap_window:
            adap_pred = np.mean(inp[-adap_window:])
        else:
            adap_pred = np.mean(inp)
            
        ex_out = {
            "input": ex['input'],
            "output": str(target),
            "metadata_step": ex.get('metadata_step', 0),
            "metadata_noise_level": ex.get('metadata_noise_level', 0.0),
            "metadata_series_length": ex.get('metadata_series_length', len(inp)),
            "predict_naive": str(naive_pred),
            "predict_fixed_ma": str(fixed_pred),
            "predict_adaptive_ma": str(adap_pred)
        }
        examples_out.append(ex_out)
        
    datasets_output.append({
        "dataset": ds_name,
        "examples": examples_out
    })

output = {
    "summary": "Adaptive moving average forecasting evaluation compared against naive and fixed MA across synthetic benchmarks.",
    "datasets": datasets_output
}
print("Evaluation complete.")

## Results and Visualization

Compare the actual targets versus the predictions from Naive, Fixed MA, and Adaptive MA models.

In [ ]:
targets = []
naive_preds = []
fixed_preds = []
adap_preds = []

for ds in datasets_output:
    for ex in ds['examples']:
        targets.append(float(ex['output']))
        naive_preds.append(float(ex['predict_naive']))
        fixed_preds.append(float(ex['predict_fixed_ma']))
        adap_preds.append(float(ex['predict_adaptive_ma']))

# Compute MSE
mse_naive = np.mean((np.array(targets) - np.array(naive_preds)) ** 2)
mse_fixed = np.mean((np.array(targets) - np.array(fixed_preds)) ** 2)
mse_adap = np.mean((np.array(targets) - np.array(adap_preds)) ** 2)

print(f"MSE - Naive: {mse_naive:.4f}")
print(f"MSE - Fixed MA: {mse_fixed:.4f}")
print(f"MSE - Adaptive MA: {mse_adap:.4f}")

# Plotting predictions vs actuals
plt.figure(figsize=(10, 5))
plt.plot(targets, label='Actual Target', marker='o', color='black', alpha=0.7)
plt.plot(naive_preds, label='Naive Forecast', linestyle='--', marker='x', color='red', alpha=0.7)
plt.plot(fixed_preds, label='Fixed MA (w=3)', linestyle='-.', marker='s', color='blue', alpha=0.7)
plt.plot(adap_preds, label='Adaptive MA', linestyle='-', marker='^', color='green', alpha=0.7)
plt.xlabel('Example Index')
plt.ylabel('Value')
plt.title('Comparison of Forecasting Methods')
plt.legend()
plt.grid(True)
plt.show()